# 04 Label Sentiment

This notebook creates and manually labels the fixed sentiment-evaluation subset.


## Fixed sample

The submitted specification requires a fixed manually labelled subset but does not prescribe its size. For this MVP, the notebook selects **300 headlines**: 30 randomly sampled headlines from each of the ten companies. The fixed random seed `2025` makes the selection reproducible and company balancing prevents firms with more GDELT coverage from dominating the labelled subset.

Labels are assigned to the target company using only information explicitly stated in the headline. This follows the entity-aware, investor-perspective approach described by [Sinha et al. (2022)](https://doi.org/10.1002/asi.24634).


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)


In [2]:
# Support running the notebook from either the project root or the pipeline directory.
working_directory = Path.cwd()
project_root = (
    working_directory.parent
    if working_directory.name == "pipeline"
    else working_directory
)

aligned_path = project_root / "data" / "processed" / "headlines_aligned_prices.csv"
gold_labels_path = project_root / "data" / "processed" / "gold_labels.csv"

print("Project root:", project_root)
print("Aligned data exists:", aligned_path.exists())


Project root: c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks
Aligned data exists: True


In [3]:
aligned_df = pd.read_csv(aligned_path)

required_columns = {
    "headline_id",
    "published_at_utc",
    "published_at_london",
    "ticker",
    "company_name",
    "headline_text",
}

assert required_columns.issubset(aligned_df.columns)
assert aligned_df["headline_id"].is_unique
assert not aligned_df[list(required_columns)].isna().any().any()
assert aligned_df["ticker"].nunique() == 10

print(f"Available aligned headlines: {len(aligned_df):,}")


Available aligned headlines: 9,001


In [4]:
SAMPLE_PER_COMPANY = 30
RANDOM_SEED = 2025

assert aligned_df.groupby("ticker").size().min() >= SAMPLE_PER_COMPANY

sample_df = (
    aligned_df.groupby("ticker", group_keys=False)
    .sample(n=SAMPLE_PER_COMPANY, random_state=RANDOM_SEED)
    .sort_values(["ticker", "published_at_utc", "headline_text"])
    .reset_index(drop=True)
)

assert len(sample_df) == 300
assert sample_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()
assert sample_df["headline_id"].is_unique

display(sample_df.groupby("ticker").size().rename("sampled_headlines"))


ticker
AZN.L     30
GSK.L     30
HLMA.L    30
HSBA.L    30
ITRK.L    30
NG.L      30
REL.L     30
RIO.L     30
SN.L      30
VOD.L     30
Name: sampled_headlines, dtype: int64

## Labelling guide

Judge the likely financial effect on the **named target company**, using only the headline. Do not use the later share price, article body, or outside knowledge.

- **positive**: the headline clearly indicates a favourable development, such as improved results, an approval, a contract win, an upgrade, or another likely benefit.
- **negative**: the headline clearly indicates an unfavourable development, such as weaker results, a warning, a rejection, litigation, disruption, a downgrade, or another likely harm.
- **neutral**: the headline is factual, mixed, unclear, or does not indicate a clear positive or negative effect on the company.

For a genuinely difficult case, choose the best provisional label and set `qa_flag` to `review`. After checking it again, clear the flag and change `annotator_pass` from `1` to `2`.

The `annotator_pass` and `qa_flag` columns are included because they are part of the submitted `gold_labels` schema.


In [5]:
display_columns = [
    "headline_id",
    "ticker",
    "company_name",
    "published_at_london",
    "headline_text",
]

labelling_df = sample_df[display_columns].copy()

if gold_labels_path.exists():
    existing_labels = pd.read_csv(gold_labels_path, keep_default_na=False)
    expected_gold_columns = [
        "headline_id", "label_gold", "annotator_pass", "qa_flag"
    ]
    assert existing_labels.columns.tolist() == expected_gold_columns
    assert existing_labels["headline_id"].is_unique
    assert set(existing_labels["headline_id"]).issubset(
        set(labelling_df["headline_id"])
    )
    labelling_df = labelling_df.merge(
        existing_labels, on="headline_id", how="left"
    )
    labelling_df["label_gold"] = labelling_df["label_gold"].fillna("")
    labelling_df["annotator_pass"] = (
        labelling_df["annotator_pass"].fillna(1).astype(int)
    )
    labelling_df["qa_flag"] = labelling_df["qa_flag"].fillna("")
else:
    labelling_df["label_gold"] = ""
    labelling_df["annotator_pass"] = 1
    labelling_df["qa_flag"] = ""

print("Existing label progress loaded.")


Existing label progress loaded.


## Enter labels

Choose one ticker at a time. The leftmost number displayed by pandas is the row index used in the editing cell below.


In [6]:
TICKER_TO_LABEL = "AZN.L"

display(
    labelling_df.loc[
        labelling_df["ticker"] == TICKER_TO_LABEL,
        [
            "ticker",
            "company_name",
            "headline_text",
            "label_gold",
            "annotator_pass",
            "qa_flag",
        ],
    ]
)


,ticker,company_name,headline_text,label_gold,annotator_pass,qa_flag
0,AZN.L,AstraZeneca plc,Trinasolar Powers AstraZeneca's Sustainability Vision with Cutting-Edge Solar Carpark Rooftop Install,,1,
1,AZN.L,AstraZeneca plc,Trinasolar Powers AstraZeneca's Sustainability Vision with Cutting-Edge Solar Carpark Rooftop Install | Taiwan News,,1,
2,AZN.L,AstraZeneca plc,ASTRAZENECA RECEIVES TWO POSITIVE NICE RECOMMENDATIONS FOR LUNG CANCER PATIENTS ACROSS ENGLAND AND WALES,,1,
3,AZN.L,AstraZeneca plc,AstraZeneca accused of rowing back on pledges in vaccine hub row,,1,
4,AZN.L,AstraZeneca plc,Covid chief attacks Labour after AstraZeneca axes its plan for £450million vaccine site,,1,
5,AZN.L,AstraZeneca plc,"Bank of England, AstraZeneca, Watches of Switzerland: Thursday ahead",,1,
6,AZN.L,AstraZeneca plc,AstraZeneca: A strong pipeline is a big tailwind - London Business News,,1,
7,AZN.L,AstraZeneca plc,Investors in AstraZeneca PLC Should Contact Levi & Korsinsky Before ...,,1,
8,AZN.L,AstraZeneca plc,Here's Why AstraZeneca PLC (AZN) Traded Lower in Q4,,1,
9,AZN.L,AstraZeneca plc,"AstraZeneca Pharma India gets CDSCO approval to import, sell cancer treatment medicine",,1,


In [7]:
# Replace the example indices with the row numbers displayed above.
# labelling_df.loc[[0, 1], "label_gold"] = "positive"
# labelling_df.loc[[2, 3], "label_gold"] = "neutral"
# labelling_df.loc[[4, 5], "label_gold"] = "negative"

# Mark a difficult case for a second check if necessary.
# labelling_df.loc[[6], "qa_flag"] = "review"


In [8]:
valid_labels = {"positive", "neutral", "negative"}
valid_qa_flags = {"", "review"}

entered_labels = set(labelling_df["label_gold"])
invalid_labels = entered_labels - valid_labels - {""}
invalid_qa_flags = set(labelling_df["qa_flag"]) - valid_qa_flags

assert not invalid_labels, f"Invalid labels: {sorted(invalid_labels)}"
assert not invalid_qa_flags, f"Invalid QA flags: {sorted(invalid_qa_flags)}"
assert labelling_df["annotator_pass"].isin([1, 2]).all()

unlabelled_count = int((labelling_df["label_gold"] == "").sum())
review_count = int((labelling_df["qa_flag"] == "review").sum())

print(f"Labelled: {len(labelling_df) - unlabelled_count:,} / {len(labelling_df):,}")
print(f"Still flagged for review: {review_count:,}")
display(labelling_df["label_gold"].value_counts(dropna=False))


Labelled: 0 / 300
Still flagged for review: 0


label_gold
    300
Name: count, dtype: int64

In [9]:
# Save progress using exactly the gold-label schema from the specification.
gold_columns = [
    "headline_id", "label_gold", "annotator_pass", "qa_flag"
]
gold_labels_df = labelling_df[gold_columns].copy()

gold_labels_path.parent.mkdir(parents=True, exist_ok=True)
gold_labels_df.to_csv(gold_labels_path, index=False)

print(f"Saved label progress to {gold_labels_path}")


Saved label progress to c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\gold_labels.csv


In [10]:
if unlabelled_count == 0 and review_count == 0:
    print("Labelling complete: gold_labels.csv is ready for model evaluation.")
else:
    print(
        "Labelling is not complete. Finish blank labels and resolve QA review flags."
    )


Labelling is not complete. Finish blank labels and resolve QA review flags.
